Version control: 2026-10-14 - 18:30
Luis Anunciacao, PhD

# DP - AGEM Time Series

Raw sources: the 10 SurveyMonkey exports in `Original data and metadata/`, read from this GitHub repository.
Output: `Importable data/df.rds`.

## §1 Records

In [1]:
pacman::p_load(readxl, tidyverse, janitor, stringi)

# downloads one SurveyMonkey export from GitHub (Original data and metadata/) and reads it with no header and every cell as text;
# private repository, token from the git credential store (gitcreds)
read_export <- function(file) {
  path <- curl::curl_download(paste0("https://raw.githubusercontent.com/anovabr/AGEM-Time-Series/main/Original%20data%20and%20metadata/", URLencode(file, reserved = TRUE)),
                              tempfile(fileext = paste0(".", tools::file_ext(file))),
                              handle = curl::new_handle() |> curl::handle_setheaders(Authorization = paste("token", gitcreds::gitcreds_get()$password)))
  if (tools::file_ext(file) == "csv") read_csv(path, col_names = FALSE, col_types = cols(.default = "c")) else read_excel(path, col_names = FALSE, col_types = "text", .name_repair = "unique_quiet")
}

# column -> item maps for the columns after the 9 SurveyMonkey metadata columns; items follow T1 numbering
# (T2-only items: lideres 21-22, emocional 50-54). c(15, 16) = one T2 column matched to two T1 items (lideres T2 "contratar e desligar");
# NA = column not used (free text of "outro"). Lideres T3 positions follow the current SurveyMonkey labels (= T1 order).
lid_t1 <- as.list(1:20)
lid_t2 <- list(1, 2, 3, 5, 4, 6, 7, 8, 9, 11, 12, 14, 13, c(15, 16), 17, 21, 18, 19, 20, 22)
lid_t3 <- c(as.list(1:20), list(23, NA, 24), as.list(rep(25, 14)))
emo_t1 <- as.list(c(1:49, rep(55, 5), 56, 57))
emo_t2 <- as.list(c(1, 2, 3, 4, 6, 7, 9, 10, 15:35, 50, 36:39, 51, 40:43, 52, 44:47, 53, 48, 54, 49, 57, 58, rep(59, 10)))

df_original <- tribble(
  ~instrumento, ~empresa, ~onda, ~arquivo, ~mapa,
  "lideres", "finx", "T1", "T1 FINX - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t1,
  "lideres", "finx", "T2", "T2 FINX - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.csv", lid_t2,
  "lideres", "finx", "T3", "T3 FINX - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.csv", lid_t3,
  "lideres", "xforce", "T1", "T1 Xforce - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t1,
  "lideres", "xforce", "T2", "T2 Xforce - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t2,
  "lideres", "xforce", "T3", "T3 XFORCE - INVENTÁRIO SAÚDE MENTAL EXCLUSIVO PARA LÍDERES.xlsx", lid_t3,
  "emocional", "finx", "T1", "T1 FINX - INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t1,
  "emocional", "finx", "T2", "T2 FINX - INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t2,
  "emocional", "xforce", "T1", "T1 Xforce- INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t1,
  "emocional", "xforce", "T2", "T2 XFORCE - INVENTÁRIO DE SAÚDE EMOCIONAL.xlsx", emo_t2) |>
  mutate(raw = map(arquivo, read_export))

rm(lid_t1, lid_t2, lid_t3, emo_t1, emo_t2)

df_original |> transmute(instrumento, empresa, onda, rows = map_int(raw, nrow), cols = map_int(raw, ncol))

instrumento,empresa,onda,rows,cols
<chr>,<chr>,<chr>,<int>,<int>
lideres,finx,T1,17,29
lideres,finx,T2,18,29
lideres,finx,T3,17,46
lideres,xforce,T1,7,29
lideres,xforce,T2,8,29
lideres,xforce,T3,5,46
emocional,finx,T1,88,65
emocional,finx,T2,88,69
emocional,xforce,T1,27,65


## §2 Analytic frame

One row per respondent × answered item, lowercased and de-accented.

- Row 1 of each export is the question text; exports with a `Response` sub-header have 2 header rows.
- `item_texto` is the wording of the earliest wave that has the item (the T3 export headers are outdated, so T3 text is used only for T3-only items).
- `codigo`: scale position (1-5) for scale items (lideres 1-22, emocional 1-54); option order for categorical items; column order for multiple-choice options.
- `codigo_3` / `resposta_3`: scale responses collapsed to 3 categories (1-2 -> 1, 3 -> 2, 4-5 -> 3: discordo / neutro / concordo; nunca/raramente / as vezes / muitas vezes/sempre); categorical and multiple-choice items keep `codigo` / `resposta`.
- `invertido`: negatively worded scale items (agreement or frequency = worse): lideres 1, 4, 5, 7, 8, 12, 18, 19, 21, 22; emocional 4-30, 39, 50-53.
- `valor_teste`: score used in the tests. Lideres T2 items 6 and 20 are worded negatively ("Não me sinto preparado(a)") and are reverse-coded (6 - codigo); counts use `codigo`.

In [2]:
likert <- list(lideres = c("discordo totalmente", "discordo parcialmente", "nem concordo nem discordo", "concordo parcialmente", "concordo totalmente"),
               emocional = c("nunca", "raramente", "as vezes", "muitas vezes", "sempre"))
likert_max <- c(lideres = 22, emocional = 54)
likert_3 <- list(lideres = c("discordo", "neutro", "concordo"), emocional = c("nunca/raramente", "as vezes", "muitas vezes/sempre"))
invertidos <- list(lideres = c(1, 4, 5, 7, 8, 12, 18, 19, 21, 22), emocional = c(4:30, 39, 50:53))
categorias <- list("lideres 23" = c("administrativo", "comercial/vendas", "financeiro", "recursos humanos", "operacional/producao", "marketing",
                                    "tecnologia/informatica", "outro (especifique)"),
                   "lideres 24" = c("menos de 3 meses", "de 3 a 6 meses", "de 6 meses a 1 ano", "de 1 a 3 anos", "de 3 a 5 anos", "mais de 5 anos"),
                   "emocional 56" = c("sim", "nao"),
                   "emocional 57" = c("ate 6 meses", "de 6 meses a 1 ano", "de 1 ano a 2 anos", "de 2 anos a 4 anos", "4 anos a 6 anos", "mais de 6 anos"),
                   "emocional 58" = c("administrativo / financeiro", "apoio / servicos gerais", "comercial / atendimento", "operacional / producao"))

# adds codigo_3 / resposta_3: scale responses collapsed to 3 categories (1-2 -> 1, 3 -> 2, 4-5 -> 3); other items keep codigo / resposta
agrupa <- \(x) x |>
  mutate(escala = item <= likert_max[instrumento],
         codigo_3 = if_else(escala, c(1L, 1L, 2L, 3L, 3L)[codigo], codigo),
         resposta_3 = if_else(escala, map2_chr(instrumento, codigo_3, \(i, k) likert_3[[i]][k]), resposta)) |>
  select(-escala)

# one export to long format: respondent x answered column, with question text (carried right over multiple-choice columns),
# item from the map and the column position inside its item (option order of multiple-choice items)
to_long <- function(raw, mapa) {
  stopifnot(ncol(raw) - 9 == length(mapa))   # the map must cover every item column
  raw |>
    slice(-seq_len(if (is.na(raw[[1]][2])) 2 else 1)) |>
    select(respondent_id = 1, 10:ncol(raw)) |>
    pivot_longer(-respondent_id, names_to = "col", values_to = "resposta", values_drop_na = TRUE) |>
    mutate(col = match(col, names(raw)) - 9,
           item_texto = (tibble(t = unlist(raw[1, -(1:9)])) |> fill(t) |> pull(t))[col],
           item = mapa[col],
           ordem = col - match(item, mapa) + 1) |>
    unnest(item) |>
    filter(!is.na(item))
}

df <- df_original |>
  mutate(long = map2(raw, mapa, to_long)) |>
  select(instrumento, empresa, onda, long) |>
  unnest(long) |>
  clean_names() |>
  mutate(across(where(is.character), ~ str_squish(stri_trans_general(tolower(.), "Latin-ASCII")))) |>
  remove_empty("rows") |>
  mutate(onda = toupper(onda)) |>
  group_by(instrumento, item) |>
  mutate(item_texto = first(item_texto[order(onda)], na_rm = TRUE)) |>
  ungroup() |>
  mutate(codigo = case_when(item <= likert_max[instrumento] ~ map2_int(resposta, instrumento, \(r, i) match(r, likert[[i]])),
                            paste(instrumento, item) %in% names(categorias) ~ map2_int(resposta, paste(instrumento, item), \(r, k) match(r, categorias[[k]])),
                            .default = as.integer(ordem)),
         valor_teste = if_else(item <= likert_max[instrumento], codigo, NA),
         valor_teste = if_else(instrumento == "lideres" & onda == "T2" & item %in% c(6, 20), 6L - valor_teste, valor_teste),
         invertido = map2_lgl(instrumento, item, \(i, it) it %in% invertidos[[i]])) |>
  agrupa() |>
  select(-col, -ordem)

stopifnot(!anyNA(df$codigo))   # every answer matched a known option

# full option list per item (scale and categorical options, including options nobody chose; multiple-choice options as observed)
attr(df, "opcoes") <- bind_rows(distinct(df, instrumento, item, codigo, resposta),
                                distinct(df, instrumento, item) |>
                                  mutate(resposta = map2(instrumento, item, \(i, it) if (it <= likert_max[i]) likert[[i]] else categorias[[paste(i, it)]])) |>
                                  unnest(resposta) |>
                                  mutate(codigo = row_number(), .by = c(instrumento, item))) |>
  distinct() |>
  agrupa() |>
  arrange(instrumento, item, codigo)

rm(likert, likert_max, likert_3, invertidos, categorias, agrupa)

df |> count(instrumento, empresa, onda, name = "answers")

instrumento,empresa,onda,answers
<chr>,<chr>,<chr>,<int>
emocional,finx,T1,4605
emocional,finx,T2,4593
emocional,xforce,T1,1334
emocional,xforce,T2,1006
lideres,finx,T1,275
lideres,finx,T2,314
lideres,finx,T3,462
lideres,xforce,T1,119
lideres,xforce,T2,125


## §3 Save

In [3]:
saveRDS(df, "../Importable data/df.rds")